# SkinSure Ensemble Evaluation
**EfficientNet-B1 + ConvNeXt-Tiny + Swin-Tiny — 5-Class Dark Skin Disease**

Melanoma | Basal_Cell_Carcinoma | Tinea | Eczema | Keratosis

## 1. Install

In [ ]:
%%capture
!pip install timm scikit-learn matplotlib seaborn
print('Done')

## 2. Imports

In [ ]:
import sys, random, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
import timm

from sklearn.metrics import (
    accuracy_score, classification_report,
    roc_auc_score, roc_curve, auc,
    confusion_matrix, matthews_corrcoef
)
from sklearn.preprocessing import label_binarize
from sklearn.calibration import calibration_curve

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True

## 3. Paths — set checkpoint paths here

In [ ]:
LOCAL_ROOT = Path('/dataset')
SAVE_DIR   = Path('/models')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

CKPT_EFF  = SAVE_DIR / 'efficientnet_b1_5cls_skin.pth'
CKPT_CNXT = SAVE_DIR / 'convnext_tiny_5cls_skin.pth'
CKPT_SWIN = SAVE_DIR / 'swin_tiny_5cls_skin.pth'

for p in [CKPT_EFF, CKPT_CNXT, CKPT_SWIN]:
    assert p.exists(), f'Checkpoint not found: {p}'
    print(f'  Found: {p}')

CLASSES = ['Basal_Cell_Carcinoma', 'Eczema', 'Keratosis', 'Melanoma', 'Tinea']
NUM_CLS  = len(CLASSES)
BATCH    = 16
WORKERS  = 1

## 4. Model Definitions

In [ ]:
class EfficientSkinClassifier(nn.Module):
    def __init__(self, num_classes, dropout=0.35, drop_path_rate=0.15):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b1', pretrained=False, num_classes=0,
            drop_rate=dropout, drop_path_rate=drop_path_rate)
        feat_dim = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.LayerNorm(feat_dim), nn.Dropout(dropout),
            nn.Linear(feat_dim, 256), nn.GELU(),
            nn.Dropout(dropout/2), nn.Linear(256, num_classes))
    def forward(self, x): return self.classifier(self.backbone(x))


class ConvNeXtSkinClassifier(nn.Module):
    def __init__(self, num_classes, dropout=0.4, drop_path_rate=0.2):
        super().__init__()
        self.backbone = timm.create_model(
            'convnext_tiny', pretrained=False, num_classes=0,
            drop_rate=dropout, drop_path_rate=drop_path_rate)
        feat_dim = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.LayerNorm(feat_dim), nn.Dropout(dropout),
            nn.Linear(feat_dim, 256), nn.GELU(),
            nn.Dropout(dropout/2), nn.Linear(256, num_classes))
    def forward(self, x): return self.classifier(self.backbone(x))


class SwinSkinClassifier(nn.Module):
    def __init__(self, model_name, num_classes, dropout=0.4, drop_path_rate=0.2):
        super().__init__()
        self.backbone = timm.create_model(
            model_name, pretrained=False, num_classes=0,
            drop_rate=dropout, drop_path_rate=drop_path_rate)
        feat_dim = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.LayerNorm(feat_dim), nn.Dropout(dropout),
            nn.Linear(feat_dim, 256), nn.GELU(),
            nn.Dropout(dropout/2), nn.Linear(256, num_classes))
    def forward(self, x): return self.classifier(self.backbone(x))


def load_model(ModelClass, path, device, **kwargs):
    ckpt  = torch.load(path, map_location=device)
    cfg   = ckpt['config']
    model = ModelClass(
        num_classes=len(ckpt['classes']),
        dropout=cfg['dropout'],
        drop_path_rate=cfg['drop_path_rate'],
        **kwargs
    )
    model.load_state_dict(ckpt['model_state'])
    model = model.to(device, memory_format=torch.channels_last).eval()
    mel   = ckpt.get('best_melanoma_sens', float('nan'))
    print(f'  val={ckpt["best_val_acc"]*100:.2f}%  '
          f'test={ckpt["test_acc"]*100:.2f}%  mel_sens={mel*100:.1f}%')
    return model, ckpt['classes']

print('Loading checkpoints...')
print('EfficientNet-B1 :', end=' ')
eff_model,  classes_eff  = load_model(EfficientSkinClassifier, CKPT_EFF,  DEVICE)
print('ConvNeXt-Tiny   :', end=' ')
cnxt_model, classes_cnxt = load_model(ConvNeXtSkinClassifier,  CKPT_CNXT, DEVICE)
print('Swin-Tiny       :', end=' ')
ckpt_swin = torch.load(CKPT_SWIN, map_location=DEVICE)
swin_model = SwinSkinClassifier(
    model_name=ckpt_swin['config']['model_name'],
    num_classes=len(ckpt_swin['classes']),
    dropout=ckpt_swin['config']['dropout'],
    drop_path_rate=ckpt_swin['config']['drop_path_rate'],
)
swin_model.load_state_dict(ckpt_swin['model_state'])
swin_model = swin_model.to(DEVICE, memory_format=torch.channels_last).eval()
mel = ckpt_swin.get('best_melanoma_sens', float('nan'))
print(f'  val={ckpt_swin["best_val_acc"]*100:.2f}%  '
      f'test={ckpt_swin["test_acc"]*100:.2f}%  mel_sens={mel*100:.1f}%')
classes_swin = ckpt_swin['classes']

assert classes_eff == classes_cnxt == classes_swin, 'Class list mismatch!'
CLASSES = classes_eff
NUM_CLS = len(CLASSES)
print(f'\nClasses: {CLASSES}')

## 5. Test DataLoader

In [ ]:
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

test_ds     = ImageFolder(LOCAL_ROOT / 'test', transform=val_tf)
test_loader = DataLoader(test_ds, batch_size=BATCH, shuffle=False,
                         num_workers=WORKERS, pin_memory=True)

print(f'Test set : {len(test_ds)} images')
print(f'Classes  : {test_ds.classes}')
assert test_ds.classes == CLASSES, (
    f'Class order mismatch!\n  Dataset: {test_ds.classes}\n  Checkpoints: {CLASSES}')

## 6. Ensemble Inference

In [ ]:
# Equal weights — change to (w_eff, w_cnxt, w_swin) to tune
WEIGHTS = (1.0, 1.0, 1.0)
w_eff, w_cnxt, w_swin = [w / sum(WEIGHTS) for w in WEIGHTS]

all_labels       = []
all_ens_probs    = []
all_eff_probs    = []
all_cnxt_probs   = []
all_swin_probs   = []
all_agreement    = []

print('Running ensemble inference...')
with torch.no_grad():
    for imgs, lbls in test_loader:
        imgs = imgs.to(DEVICE, memory_format=torch.channels_last, non_blocking=True)

        p_eff  = F.softmax(eff_model(imgs),  dim=1).cpu().numpy()
        p_cnxt = F.softmax(cnxt_model(imgs), dim=1).cpu().numpy()
        p_swin = F.softmax(swin_model(imgs), dim=1).cpu().numpy()

        p_ens = w_eff * p_eff + w_cnxt * p_cnxt + w_swin * p_swin

        ens_top  = p_ens.argmax(axis=1)
        eff_top  = p_eff.argmax(axis=1)
        cnxt_top = p_cnxt.argmax(axis=1)
        swin_top = p_swin.argmax(axis=1)

        agreement = (
            (eff_top == ens_top).astype(float) +
            (cnxt_top == ens_top).astype(float) +
            (swin_top == ens_top).astype(float)
        ) / 3.0

        all_labels.extend(lbls.numpy())
        all_ens_probs.extend(p_ens)
        all_eff_probs.extend(p_eff)
        all_cnxt_probs.extend(p_cnxt)
        all_swin_probs.extend(p_swin)
        all_agreement.extend(agreement)

all_labels     = np.array(all_labels)
all_ens_probs  = np.array(all_ens_probs)
all_eff_probs  = np.array(all_eff_probs)
all_cnxt_probs = np.array(all_cnxt_probs)
all_swin_probs = np.array(all_swin_probs)
all_agreement  = np.array(all_agreement)

all_ens_preds  = all_ens_probs.argmax(axis=1)
all_eff_preds  = all_eff_probs.argmax(axis=1)
all_cnxt_preds = all_cnxt_probs.argmax(axis=1)
all_swin_preds = all_swin_probs.argmax(axis=1)

print('Done.')

## 7. Per-Model vs Ensemble Accuracy Table

In [ ]:
y_bin     = label_binarize(all_labels, classes=list(range(NUM_CLS)))

models = {
    'EfficientNet-B1' : (all_eff_preds,  all_eff_probs),
    'ConvNeXt-Tiny'   : (all_cnxt_preds, all_cnxt_probs),
    'Swin-Tiny'       : (all_swin_preds, all_swin_probs),
    'Ensemble'        : (all_ens_preds,  all_ens_probs),
}

print(f'{"Model":<20} {"Accuracy":>10} {"Macro-AUC":>12} {"MCC":>8}')
print('-' * 54)
summary = {}
for name, (preds, probs) in models.items():
    acc  = accuracy_score(all_labels, preds)
    mauc = roc_auc_score(y_bin, probs, multi_class='ovr', average='macro')
    mcc  = matthews_corrcoef(all_labels, preds)
    summary[name] = dict(acc=acc, mauc=mauc, mcc=mcc)
    marker = ' ←' if name == 'Ensemble' else ''
    print(f'{name:<20} {acc*100:>9.2f}% {mauc:>12.4f} {mcc:>8.4f}{marker}')

## 8. Clinical Sensitivity & Specificity Table

In [ ]:
CLINICAL_TARGETS = {
    'Melanoma'            : 0.95,
    'Basal_Cell_Carcinoma': 0.90,
    'Tinea'               : 0.80,
    'Eczema'              : 0.80,
    'Keratosis'           : 0.80,
}

print('Ensemble — Clinical Performance')
print('=' * 72)
print(f'{"Class":<25} {"Sens":>6} {"Spec":>6} {"PPV":>6} {"NPV":>6} {"AUC":>6} {"Target":>7} {"":>4}')
print('=' * 72)

clinical_rows = []
for i, cls in enumerate(CLASSES):
    tp = ((all_ens_preds==i) & (all_labels==i)).sum()
    fn = ((all_ens_preds!=i) & (all_labels==i)).sum()
    tn = ((all_ens_preds!=i) & (all_labels!=i)).sum()
    fp = ((all_ens_preds==i) & (all_labels!=i)).sum()
    sens = tp/(tp+fn) if (tp+fn)>0 else 0
    spec = tn/(tn+fp) if (tn+fp)>0 else 0
    ppv  = tp/(tp+fp) if (tp+fp)>0 else 0
    npv  = tn/(tn+fn) if (tn+fn)>0 else 0
    auc_s = roc_auc_score(y_bin[:,i], all_ens_probs[:,i])
    target = CLINICAL_TARGETS[cls]
    status = 'PASS' if sens >= target else 'FAIL'
    clinical_rows.append(dict(cls=cls, sens=sens, spec=spec, ppv=ppv, npv=npv, auc=auc_s))
    print(f'{cls:<25} {sens:>6.3f} {spec:>6.3f} {ppv:>6.3f} {npv:>6.3f} '
          f'{auc_s:>6.3f} {target:>6.0%}  {status}')
print('=' * 72)

macro_auc = roc_auc_score(y_bin, all_ens_probs, multi_class='ovr', average='macro')
ens_acc   = accuracy_score(all_labels, all_ens_preds)
print(f'\nEnsemble Test Accuracy : {ens_acc*100:.2f}%')
print(f'Ensemble Macro AUC     : {macro_auc:.4f}')
print(f'Mean Agreement Score   : {all_agreement.mean():.3f}')

## 9. Classification Report

In [ ]:
print(classification_report(all_labels, all_ens_preds, target_names=CLASSES, digits=4))

## 10. Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_ens_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0],
            linewidths=0.5, linecolor='gray')
axes[0].set_title('Confusion Matrix — Counts', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predicted', fontsize=11)
axes[0].set_ylabel('True', fontsize=11)
axes[0].tick_params(axis='x', rotation=30)

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[1],
            vmin=0, vmax=1, linewidths=0.5, linecolor='gray')
axes[1].set_title('Confusion Matrix — Normalised', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Predicted', fontsize=11)
axes[1].set_ylabel('True', fontsize=11)
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle(f'SkinSure Ensemble (EfficientNet + ConvNeXt + Swin) | Acc={ens_acc*100:.2f}%',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(SAVE_DIR / 'ensemble_confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved: ensemble_confusion_matrix.png')

## 11. ROC Curves — Ensemble vs Individual Models

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()
colors = plt.cm.tab10.colors

for i, cls in enumerate(CLASSES):
    ax = axes[i]
    for model_name, probs, ls in [
        ('EfficientNet-B1', all_eff_probs,  '--'),
        ('ConvNeXt-Tiny',   all_cnxt_probs, '-.'),
        ('Swin-Tiny',       all_swin_probs, ':'),
        ('Ensemble',        all_ens_probs,  '-'),
    ]:
        fpr, tpr, _ = roc_curve(y_bin[:, i], probs[:, i])
        auc_val = auc(fpr, tpr)
        lw = 2.5 if model_name == 'Ensemble' else 1.5
        ax.plot(fpr, tpr, linestyle=ls, linewidth=lw,
                label=f'{model_name} (AUC={auc_val:.3f})')
    ax.plot([0,1],[0,1],'k--', lw=1, alpha=0.4)
    ax.set_title(cls, fontweight='bold', fontsize=11)
    ax.set_xlabel('False Positive Rate', fontsize=9)
    ax.set_ylabel('True Positive Rate', fontsize=9)
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(alpha=0.3)

# Last panel: macro-average ROC
ax = axes[5]
for model_name, probs, ls in [
    ('EfficientNet-B1', all_eff_probs,  '--'),
    ('ConvNeXt-Tiny',   all_cnxt_probs, '-.'),
    ('Swin-Tiny',       all_swin_probs, ':'),
    ('Ensemble',        all_ens_probs,  '-'),
]:
    mauc = roc_auc_score(y_bin, probs, multi_class='ovr', average='macro')
    lw = 2.5 if model_name == 'Ensemble' else 1.5
    # plot mean FPR/TPR for macro curve
    mean_fpr = np.linspace(0, 1, 200)
    tprs = []
    for j in range(NUM_CLS):
        fpr_j, tpr_j, _ = roc_curve(y_bin[:, j], probs[:, j])
        tprs.append(np.interp(mean_fpr, fpr_j, tpr_j))
    mean_tpr = np.mean(tprs, axis=0)
    ax.plot(mean_fpr, mean_tpr, linestyle=ls, linewidth=lw,
            label=f'{model_name} (macro={mauc:.3f})')
ax.plot([0,1],[0,1],'k--', lw=1, alpha=0.4)
ax.set_title('Macro-Average ROC', fontweight='bold', fontsize=11)
ax.set_xlabel('False Positive Rate', fontsize=9)
ax.set_ylabel('True Positive Rate', fontsize=9)
ax.legend(fontsize=8, loc='lower right')
ax.grid(alpha=0.3)

plt.suptitle('ROC Curves — SkinSure Ensemble vs Individual Backbones',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(SAVE_DIR / 'ensemble_roc_curves.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved: ensemble_roc_curves.png')

## 12. Per-Model Accuracy Bar Chart

In [ ]:
model_names = ['EfficientNet-B1', 'ConvNeXt-Tiny', 'Swin-Tiny', 'Ensemble']
accs  = [summary[m]['acc']*100  for m in model_names]
maucs = [summary[m]['mauc']     for m in model_names]
mccs  = [summary[m]['mcc']      for m in model_names]

x = np.arange(len(model_names))
bar_colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, vals, ylabel, title, fmt in zip(
    axes,
    [accs, maucs, mccs],
    ['Accuracy (%)', 'Macro AUC', 'MCC'],
    ['Test Accuracy', 'Macro-Average AUC', 'Matthews Correlation Coefficient'],
    ['{:.2f}%', '{:.4f}', '{:.4f}']
):
    bars = ax.bar(x, vals, color=bar_colors, edgecolor='black', linewidth=0.7, width=0.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(vals)*0.01,
                fmt.format(val), ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(model_names, rotation=15, ha='right', fontsize=9)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_ylim(0, max(vals)*1.12)
    ax.grid(axis='y', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Model Comparison — SkinSure Ensemble vs Individual Backbones',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(SAVE_DIR / 'ensemble_model_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved: ensemble_model_comparison.png')

## 13. Agreement Score Distribution

In [ ]:
agree_vals, agree_counts = np.unique(all_agreement, return_counts=True)
agree_labels = {0.33: 'One agrees\n(0.33)', 0.67: 'Two agree\n(0.67)', 1.0: 'All agree\n(1.0)'}

# Accuracy conditioned on agreement tier
tiers = {}
for val in [0.33, 0.67, 1.0]:
    mask = all_agreement == val
    if mask.sum() > 0:
        tier_acc = accuracy_score(all_labels[mask], all_ens_preds[mask])
        tiers[val] = (mask.sum(), tier_acc)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Distribution bar
ax = axes[0]
bar_x = [agree_labels.get(v, str(v)) for v in agree_vals]
bars = ax.bar(bar_x, agree_counts, color=['#C44E52','#DD8452','#55A868'],
              edgecolor='black', linewidth=0.7, width=0.4)
for bar, cnt in zip(bars, agree_counts):
    pct = cnt / len(all_agreement) * 100
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
            f'{cnt}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Agreement Score Distribution', fontweight='bold', fontsize=12)
ax.set_ylabel('Number of Test Images', fontsize=10)
ax.set_ylim(0, max(agree_counts)*1.18)
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# Accuracy by tier
ax = axes[1]
tier_keys   = [0.33, 0.67, 1.0]
tier_accs   = [tiers[k][1]*100 for k in tier_keys if k in tiers]
tier_labels = [agree_labels[k] for k in tier_keys if k in tiers]
bars = ax.bar(tier_labels, tier_accs, color=['#C44E52','#DD8452','#55A868'],
              edgecolor='black', linewidth=0.7, width=0.4)
for bar, val in zip(bars, tier_accs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'{val:.2f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Ensemble Accuracy by Agreement Tier', fontweight='bold', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=10)
ax.set_ylim(0, 115)
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.suptitle('Ensemble Model Agreement Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(SAVE_DIR / 'ensemble_agreement.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved: ensemble_agreement.png')

## 14. Per-Class Sensitivity Radar Chart

In [ ]:
sensitivities = {}
for name, preds in [
    ('EfficientNet-B1', all_eff_preds),
    ('ConvNeXt-Tiny',   all_cnxt_preds),
    ('Swin-Tiny',       all_swin_preds),
    ('Ensemble',        all_ens_preds),
]:
    sens_list = []
    for i in range(NUM_CLS):
        tp = ((preds==i) & (all_labels==i)).sum()
        fn = ((preds!=i) & (all_labels==i)).sum()
        sens_list.append(tp/(tp+fn) if (tp+fn)>0 else 0)
    sensitivities[name] = sens_list

angles = np.linspace(0, 2*np.pi, NUM_CLS, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
colors_radar = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']
styles = ['--', '-.', ':', '-']
lwidths = [1.5, 1.5, 1.5, 2.5]

for (name, vals), color, ls, lw in zip(sensitivities.items(), colors_radar, styles, lwidths):
    vals_plot = vals + vals[:1]
    ax.plot(angles, vals_plot, color=color, linestyle=ls, linewidth=lw, label=name)
    ax.fill(angles, vals_plot, color=color, alpha=0.05)

# Clinical target rings
for target, label in [(0.80, '80% target'), (0.90, '90% target'), (0.95, '95% target')]:
    ax.plot(angles, [target]*(NUM_CLS+1), 'gray', linestyle=':', linewidth=0.8, alpha=0.6)
    ax.text(angles[0], target+0.02, label, fontsize=7, color='gray', ha='center')

ax.set_xticks(angles[:-1])
ax.set_xticklabels(CLASSES, fontsize=10)
ax.set_ylim(0, 1.05)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['20%','40%','60%','80%','100%'], fontsize=8)
ax.set_title('Per-Class Sensitivity — Ensemble vs Backbones',
             fontsize=13, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(str(SAVE_DIR / 'ensemble_sensitivity_radar.png'), dpi=300, bbox_inches='tight')
plt.show()
print('Saved: ensemble_sensitivity_radar.png')

## 15. Summary — All Saved Files

In [ ]:
saved = [
    'ensemble_confusion_matrix.png',
    'ensemble_roc_curves.png',
    'ensemble_model_comparison.png',
    'ensemble_agreement.png',
    'ensemble_sensitivity_radar.png',
]
print(f'All outputs saved to: {SAVE_DIR}')
for f in saved:
    p = SAVE_DIR / f
    size = p.stat().st_size // 1024 if p.exists() else 0
    status = 'OK' if p.exists() else 'MISSING'
    print(f'  [{status}] {f:<45} {size} KB')

print(f'\nFinal ensemble results:')
print(f'  Test Accuracy : {ens_acc*100:.2f}%')
print(f'  Macro AUC     : {macro_auc:.4f}')
print(f'  MCC           : {summary["Ensemble"]["mcc"]:.4f}')
print(f'  Mean Agreement: {all_agreement.mean():.3f}')